# Mass reconstruction and two 2D efficiency maps

Panel **(a)** retains the three-state mass fit, legend, fitted yields and Gaussian-core resolutions from notebook01. Panels **(b)** and **(c)** show the saved **1S** proton-tagged efficiencies versus $p_{T,\Upsilon}$ and $y_\Upsilon$, and versus $x_V$ and $Q^2$. The lower panels have no top campaign/state/energy/Q² annotations, legends or hatching. Both colorbars read **Efficiency [%]**.

Place this notebook **beside `01_reconstruction.ipynb`**, with the existing `upsilon_esr/` folder and `settings.py`. Set your completed full-framework and 2D run IDs in the shared plotting profile, then **Run All**. It reads numerical CSV/JSON outputs and repeats the same notebook01 mass fit; it does not rerun the framework or read ROOT. Each execution writes a fresh combined-figure directory and preserves previous files.

The mass fit is conditional on the calibrated signal shapes and has no continuum background. Its quoted resolutions are DSCB Gaussian-core widths, not RMS values. The mass yield is projected to the shared profile luminosity (2.5 fb⁻¹ in the example). The 1S efficiency maps use raw MC counts with no luminosity scaling. The current 2S/3S mass inputs retain their different Q²/ID settings. Gray map cells have no eligible MC. These selections and the required proton tag remain in the audit even though the lower-panel headers are hidden.


**Repository profile:** copy `configs/plots_9x275.example.json` to `configs/plots.local.json` and edit that one file. Paths, runs, luminosity, selections, optional `display_rebin` and diagnostic permission are shared by all notebooks. See `docs/plotting.md`. Every execution creates a fresh output directory. The current figure layouts require all three states; the 1S-only 9×130 support run is reviewed through its status/audit outputs.


In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import tempfile
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink
from settings import load_profile, make_config, new_output_dir, map_source, file_link_path
from upsilon_esr import preflight

NOTEBOOK_DIR = Path.cwd().resolve()
PROFILE = load_profile()
OUTPUT_DIR = new_output_dir(PROFILE, "mass_and_2d_efficiency")
cfg = make_config(profile=PROFILE, output_dir=OUTPUT_DIR)
MAP_SAMPLE = PROFILE["map_sample"]
MAP_SOURCE = map_source(PROFILE)
print("Profile:", PROFILE["profile_path"])
print("Output directory:", OUTPUT_DIR)


The following collapsed cell contains the layout and saved-input checks. The existing notebook01 fit implementation is reused.

In [ ]:
def _grid(rows):
    """Rebuild the exact stored count grid, retaining undefined cells."""
    if not rows:
        raise ValueError("No numerical rows supplied for an efficiency map")
    x = np.array(sorted({float(r[k]) for r in rows for k in ("x_low", "x_high")}))
    y = np.array(sorted({float(r[k]) for r in rows for k in ("y_low", "y_high")}))
    if not (np.all(np.isfinite(x)) and np.all(np.isfinite(y))):
        raise ValueError("Nonfinite map edges")
    n = np.zeros((len(y)-1, len(x)-1), dtype=np.int64)
    k = np.zeros_like(n)
    seen = np.zeros_like(n, dtype=bool)
    for row in rows:
        ix, iy = (int(np.searchsorted(v, float(row[name])))
                  for v, name in ((x, "x_low"), (y, "y_low")))
        if (ix >= len(x)-1 or iy >= len(y)-1
                or x[ix+1] != float(row["x_high"])
                or y[iy+1] != float(row["y_high"])):
            raise ValueError("Map cells must tile adjacent edges")
        if seen[iy, ix]:
            raise ValueError("Duplicate map cell")
        ni, ki = float(row["denominator"]), float(row["selected"])
        if not (np.isfinite(ni) and np.isfinite(ki) and ni.is_integer()
                and ki.is_integer() and 0 <= ki <= ni):
            raise ValueError("Map counts require integers with 0 <= selected <= denominator")
        if ni > 0 and not np.isclose(float(row["efficiency"]), ki/ni, rtol=0, atol=1e-12):
            raise ValueError("Saved efficiency disagrees with the raw MC counts")
        n[iy,ix], k[iy,ix], seen[iy,ix] = int(ni), int(ki), True
    if not seen.all():
        raise ValueError("Incomplete stored map grid")
    values = np.divide(k, n, out=np.zeros_like(n, dtype=float), where=n > 0)
    return x, y, n, np.ma.array(100*values, mask=n == 0)


def _same_selection(left, right, name):
    a, b = np.asarray(left, dtype=float), np.asarray(right, dtype=float)
    if a.shape != b.shape or not np.isfinite(a).all() or not np.allclose(a,b,rtol=0,atol=1e-9):
        raise ValueError(f"Mass and efficiency inputs disagree on {name}: {left} versus {right}")


def validate_map_inputs(cfg, source, sample_key):
    from upsilon_esr.common import resolve_artifacts, sample_metadata
    table_path = source / "tables/final_efficiency_2d.csv"
    audit_path = source / "selection_audit.json"
    for path in (table_path, audit_path):
        if not path.is_file():
            raise FileNotFoundError(f"Missing saved 2D output: {path}. Check MAP_RUN_ID and MAP_SAMPLE.")
    audit = json.loads(audit_path.read_text())
    sample, settings = audit["config"]["sample"], audit["config"]["settings"]
    if sample["key"] != sample_key or sample["state"] != "1S":
        raise ValueError("Panels (b) and (c) require the selected 1S map sample.")
    if audit.get("status") != "COMPLETE":
        raise ValueError("Saved efficiency audit does not describe a completed calculation.")
    if settings.get("proton_selection") != "required" or audit.get("proton_selection",{}).get("policy") != "required":
        raise ValueError("The 2D maps must use the required proton tag.")
    energy = f"{settings['electron_energy']:g}x{settings['proton_energy']:g}"
    if energy != cfg.energy:
        raise ValueError("Mass and efficiency inputs have different beam energies.")
    if audit.get("weights_used") is not False or audit.get("luminosity_used") is not False:
        raise ValueError("Expected MC-count efficiencies without weights or luminosity scaling.")
    id_labels = {"new":"new electron ID", "standard":"standard electron ID"}
    if id_labels.get(sample.get("id_mode")) != cfg.categories["1S"]:
        raise ValueError("Mass and map configured electron-ID categories disagree.")
    validated = sample_metadata(cfg,"1S")
    if audit["proton_selection"].get("predicate") != validated["proton_selection"]["predicate"]:
        raise ValueError("Mass and map proton-tag predicates disagree.")
    perf_path = resolve_artifacts(cfg.runs["1S"]) / "performance" / cfg.energy / "performance_audit.json"
    perf = json.loads(perf_path.read_text())
    for key in ("Q2_GeV2", "W_GeV"):
        _same_selection(audit["truth_selection"][key],perf["truth_selection"][key],"truth "+key)
    _same_selection(audit["truth_selection"]["mass_GeV"],perf["truth_selection"]["mass_window_GeV"],"truth mass window")
    for key in ("mass_GeV", "W_GeV", "sigma_epz_GeV"):
        _same_selection(audit["reco_selection"][key],perf["reco_selection"][key],"reconstructed "+key)
    _same_selection(sample["q2_max"],cfg.q2_max["1S"],"configured Q2 maximum")
    with table_path.open(newline="") as stream:
        rows = list(csv.DictReader(stream))
    if not rows or {r.get("sample") for r in rows} != {sample_key}:
        raise ValueError("The map CSV is empty or mixes samples.")
    if {r.get("proton_selection") for r in rows} != {"required"}:
        raise ValueError("The map CSV does not consistently record proton tagging.")
    count_checks = {}
    for pkey,mkey in (("tuple_rows","raw_rows"),("eligible_truth_rows","eligible_truth"),("selected_rows","selected")):
        if pkey in perf and mkey in audit:
            if int(perf[pkey]) != int(audit[mkey]):
                raise ValueError(f"The 1S mass and map raw counts disagree: {pkey} versus {mkey}.")
            count_checks[pkey] = {"status":"PASS", "count":int(perf[pkey])}
        else:
            count_checks[pkey] = {"status":"NOT_CHECKED", "reason":"One source does not record this raw count"}
    grids = {name:_grid([r for r in rows if r["map"]==name]) for name in ("pt_y","xv_q2")}
    if np.any(grids["xv_q2"][0] <= 0) or np.any(grids["xv_q2"][1] <= 0):
        raise ValueError("The xV-Q2 map requires strictly positive log-axis edges.")
    return rows, audit, grids, {
        "selection_agreement":"PASS", "raw_count_checks":count_checks,
        "map_state":"1S", "mass_panel_states":["1S","2S","3S"],
        "sample_identity":"Matching metadata/counts do not authenticate the producer revision or prove event-by-event identity.",
        "sources":[{"path":str(p),"sha256":hashlib.sha256(p.read_bytes()).hexdigest()}
                   for p in (table_path,audit_path,perf_path)]}


def draw_combined(cfg, grids, map_audit, consistency, output_dir):
    from dataclasses import replace
    from matplotlib.colors import Normalize
    from matplotlib.ticker import MaxNLocator, LogLocator, NullFormatter
    from upsilon_esr.reconstruction import plot_reconstruction
    from upsilon_esr.common import write_json
    mass_cfg = replace(cfg, output_dir=output_dir / "mass_reference")
    mass_cfg.synthetic = bool(cfg.synthetic or map_audit.get("synthetic",False))
    # Reuse notebook01's exact fit and its existing top-panel artists.
    fig, mass_audit = plot_reconstruction(mass_cfg, rebin=2,
        float_common_shape=True, efficiency_variables=("true_W","true_pT"), show_mass_intervals=False)
    mass_ax = fig.axes[0]
    for old_axis in list(fig.axes[1:]):
        old_axis.remove()
    fig.set_size_inches(9.6, 8.8)
    layout = fig.add_gridspec(2,2, height_ratios=(1.12,1),
        left=.10,right=.93,bottom=.09,top=.96,hspace=.43,wspace=.50)
    mass_ax.set_subplotspec(layout[0,:])
    # Keep the mass curve, table and legend intact; align the panel identifier.
    for text in mass_ax.texts:
        if text.get_text()=="(a)":
            text.set_position((0,1.035));text.set_ha("left");text.set_va("bottom")
    definitions = [
        ("pt_y", "b", r"$p_{T,\Upsilon}^{\mathrm{true}}$ [GeV]",r"$y_{\Upsilon}^{\mathrm{true}}$",False),
        ("xv_q2", "c", r"$x_V^{\mathrm{true}}$",r"$Q^{2}_{\mathrm{true}}$ [GeV$^2$]",True),
    ]
    map_axes=[]
    for column,(name,panel,xlabel,ylabel,logarithmic) in enumerate(definitions):
        inner=layout[1,column].subgridspec(1,2,width_ratios=(1,.055),wspace=.10)
        ax=fig.add_subplot(inner[0,0]);cax=fig.add_subplot(inner[0,1])
        x,y,n,efficiency=grids[name]
        cmap=plt.get_cmap("viridis").copy();cmap.set_bad("#eeeeee")
        mesh=ax.pcolormesh(x,y,efficiency,cmap=cmap,norm=Normalize(0,100),
                           shading="flat",rasterized=True)
        if logarithmic:
            ax.set_xscale("log");ax.set_yscale("log")
            ax.xaxis.set_major_locator(LogLocator(base=10,numticks=4))
            ax.yaxis.set_major_locator(LogLocator(base=10,numticks=6))
            ax.xaxis.set_minor_formatter(NullFormatter())
            ax.yaxis.set_minor_formatter(NullFormatter())
        else:
            ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
        ax.set_xlim(x[0],x[-1]);ax.set_ylim(y[0],y[-1])
        ax.set_xlabel(xlabel,fontsize=15,labelpad=7)
        ax.set_ylabel(ylabel,fontsize=15,labelpad=7)
        ax.tick_params(which="both",labelsize=12.5,top=True,right=True,direction="in")
        # No campaign, state, energy or Q2 headings in the lower panels.
        ax.text(0,1.035,f"({panel})",transform=ax.transAxes,ha="left",va="bottom",fontsize=15)
        colorbar=fig.colorbar(mesh,cax=cax,ticks=np.arange(0,101,20))
        colorbar.set_label("Efficiency [%]",fontsize=13,labelpad=8)
        colorbar.ax.tick_params(labelsize=11,direction="in")
        map_axes.append(ax)
    stem="upsilon_mass_and_2d_efficiency"
    paths=[]
    for ext in ("pdf","png"):
        path=output_dir/(stem+"."+ext)
        fig.savefig(path,bbox_inches="tight",pad_inches=.12,dpi=220)
        paths.append(str(path))
    metadata={
        "figure":stem,"energy":cfg.energy,"mass_luminosity_invfb":float(cfg.lumi_invfb),
        "panels":{"a":"Three-state expected mass spectrum and conditional DSCB fit",
                  "b":"1S MC efficiency, true pT horizontal and true meson rapidity vertical",
                  "c":"1S MC efficiency, true xV horizontal and true Q2 vertical"},
        "lower_panel_top_annotations":False,"hatching":False,"colorbar":"Efficiency [%]",
        "mass_fit_method_changed":False,"map_luminosity_scaling_applied":False,
        "mass_audit":mass_audit,"map_audit":map_audit,"input_consistency":consistency,
        "synthetic":mass_cfg.synthetic,"outputs":paths,
    }
    # Fingerprints of mass inputs are numerical source files, never ROOT rereads.
    metadata["mass_input_fingerprints"]=[{"path":p,"sha256":hashlib.sha256(Path(p).read_bytes()).hexdigest()}
        for paths_for_state in mass_audit["input_files"].values() for p in paths_for_state]
    write_json(output_dir/(stem+"_audit.json"),metadata)
    return fig,metadata,map_axes


In [ ]:
display(preflight(cfg,section="reconstruction"))
map_rows,map_audit,map_grids,consistency = validate_map_inputs(cfg,MAP_SOURCE,MAP_SAMPLE)
print("Mass/map 1S selection agreement:",consistency["selection_agreement"])
print("Lower panels:",MAP_SAMPLE,"— efficiencies are not luminosity scaled")


In [ ]:
fig,audit,map_axes = draw_combined(cfg,map_grids,map_audit,consistency,OUTPUT_DIR)
plt.close(fig)
display(Image(filename=str(OUTPUT_DIR / "upsilon_mass_and_2d_efficiency.png"),width=960))


In [ ]:
print("Output directory:",OUTPUT_DIR)
for extension in ("png","pdf"):
    path=OUTPUT_DIR / ("upsilon_mass_and_2d_efficiency."+extension)
    display(FileLink(file_link_path(path),result_html_prefix=extension.upper()+": "))
display(FileLink(file_link_path(OUTPUT_DIR / "upsilon_mass_and_2d_efficiency_audit.json"),result_html_prefix="Audit: "))
